# Phase 2C — Vectorisation vs loops vs Numba

Instructor starter: [vectorise_vs_loop.py](https://github.com/adaskin/hpc-for-ai-course/blob/main/projects/project1-hp-ss-engine/starter_code/phase2_matrix_ops/vectorise_vs_loop.py). Implement the **same Gram matrix** `G=X.T @ X` using four approaches. Input is a subset of our **real 20K × 384 MiniLM embeddings** (not synthetic). CPU-only benchmark. **PILOT=True first**; then full benchmark. Outputs saved to Drive and later mirrored to GitHub. No invented timings.

Important: the instructor's skip rule checks `m*n` but the loop actually performs `m*n*n` operations; this notebook uses the correct workload estimate. Never infer a speedup against pure Python for workloads where Python was not actually timed.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time, gc
from pathlib import Path
from numba import njit, prange, set_num_threads, get_num_threads
from threadpoolctl import threadpool_limits
from google.colab import drive

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
OUT=ROOT/'phase2_matrix_ops'
DATA=OUT/'2A_sparse_dense'/'qwen_20k_minilm_384_float32.npy'
assert DATA.exists(),f'Missing cached embeddings: {DATA}'
X=np.load(DATA,mmap_mode='r')
assert X.shape==(20000,384)
(OUT/'benchmarks').mkdir(parents=True,exist_ok=True)
(OUT/'figures').mkdir(parents=True,exist_ok=True)
print('Real embeddings',X.shape)

## Four implementations

All functions compute the same `n × n` Gram matrix. Python is intentionally slow. Numba compiles loops and the parallel version uses `prange` over output rows. NumPy uses optimized BLAS. We use float64 to match starter-code numerical behavior.

In [ ]:
def gram_elementwise(A):
    m,n=A.shape
    G=np.zeros((n,n),dtype=np.float64)
    for i in range(n):
        for j in range(n):
            s=0.0
            for t in range(m):
                s+=A[t,i]*A[t,j]
            G[i,j]=s
    return G

def gram_vectorised(A):
    return A.T@A

@njit
def gram_numba_serial(A):
    m,n=A.shape
    G=np.zeros((n,n),dtype=np.float64)
    for i in range(n):
        for j in range(n):
            s=0.0
            for t in range(m):
                s+=A[t,i]*A[t,j]
            G[i,j]=s
    return G

@njit(parallel=True)
def gram_numba_parallel(A):
    m,n=A.shape
    G=np.zeros((n,n),dtype=np.float64)
    for i in prange(n):
        for j in range(n):
            s=0.0
            for t in range(m):
                s+=A[t,i]*A[t,j]
            G[i,j]=s
    return G

A=np.ascontiguousarray(X[:100,:32],dtype=np.float64)
# Warm-up JIT before correctness and timing
_ = gram_numba_serial(A)
_ = gram_numba_parallel(A)
reference=gram_elementwise(A)
for name,fn in [('NumPy',gram_vectorised),('Numba serial',gram_numba_serial),('Numba parallel',gram_numba_parallel)]:
    got=fn(A)
    assert np.allclose(reference,got,rtol=1e-5,atol=1e-9),name
print('All four implementations agree on real embeddings')

## Pilot and full workloads

Pilot: 100×32, 300×64. Full: 100×32, 300×64, 1000×96, 5000×128, 5000×300 (instructor reference size). Python loops run **only if m*n*n <= 2,000,000**; otherwise their results are marked skipped. NumPy/Numba always run. Warm-up compilation is excluded. BLAS threads are limited to 2; Numba uses up to 2 threads to keep comparisons manageable on Colab CPU. Repeat 3 times in pilot, 5 in full.

In [ ]:
PILOT=True
WORKLOADS=[(100,32),(300,64)] if PILOT else [(100,32),(300,64),(1000,96),(5000,128),(5000,300)]
REPEATS=3 if PILOT else 5
MAX_PYTHON_OPS=2_000_000
THREADS=2
set_num_threads(min(THREADS,get_num_threads()))
print('Numba threads',get_num_threads())
methods=[('elementwise',gram_elementwise),('vectorised',gram_vectorised),('numba_serial',gram_numba_serial),('numba_parallel',gram_numba_parallel)]
rows=[]
with threadpool_limits(limits=THREADS):
    for m,n in WORKLOADS:
        A=np.ascontiguousarray(X[:m,:n],dtype=np.float64)
        workload=m*n*n
        for name,fn in methods:
            if name=='elementwise' and workload>MAX_PYTHON_OPS:
                print('SKIP Python loops',m,n,'estimated operations',workload,flush=True)
                continue
            # JIT warm-up and optional cache warming, not included in timing
            if name.startswith('numba'): fn(A)
            for repeat in range(REPEATS):
                t0=time.perf_counter();G=fn(A);elapsed=time.perf_counter()-t0
                rows.append(dict(rows=m,columns=n,method=name,repeat=repeat,seconds=elapsed,estimated_ops=workload,threads=THREADS))
            print('Finished',m,n,name,flush=True)
        gc.collect()
raw=pd.DataFrame(rows)
suffix='pilot' if PILOT else 'full'
path=OUT/'benchmarks'/f'phase2c_gram_timings_{suffix}.csv'
raw.to_csv(path,index=False)
print('Saved',path,'measurements',len(raw))
print(raw.groupby(['rows','columns','method']).seconds.median().round(6))

## Real speedup plot and scaling

Speedups vs Python loops are displayed **only for workloads where the Python baseline was measured**. For larger workloads, compare NumPy and Numba directly; do not extrapolate a Python baseline or claim a 1000× improvement without measurements.

In [ ]:
summary=raw.groupby(['rows','columns','method'],as_index=False).agg(median_s=('seconds','median'),mean_s=('seconds','mean'),std_s=('seconds','std'))
summary.to_csv(OUT/'benchmarks'/f'phase2c_gram_summary_{suffix}.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(13,5))
for method,g in summary.groupby('method'):
    g=g.sort_values(['rows','columns'])
    axes[0].plot([f'{int(x)}x{int(y)}' for x,y in zip(g.rows,g.columns)],g.median_s,'o-',label=method)
axes[0].set(yscale='log',xlabel='Matrix shape',ylabel='Median seconds (log scale)',title='Gram matrix compute time')
for (m,n),g in summary.groupby(['rows','columns']):
    baseline=g[g.method=='elementwise']
    if len(baseline):
        b=float(baseline.median_s.iloc[0])
        for _,v in g.iterrows():
            if v.method!='elementwise':axes[1].scatter(f'{m}x{n}',b/v.median_s,label=v.method if (m,n)==WORKLOADS[0] else None)
axes[1].set(xlabel='Matrix shape (Python baseline measured)',ylabel='Speedup vs Python loops',title='Measured speedups only')
for ax in axes:ax.grid(alpha=.25);ax.legend(fontsize=8)
fig.tight_layout();p=OUT/'figures'/f'phase2c_gram_{suffix}.png'
fig.savefig(p,dpi=150,bbox_inches='tight');plt.show();print('Saved',p)
print('NumPy fast because optimized BLAS avoids Python interpreter overhead; Numba helps loops after compilation. Parallel speedup limited by cores, scheduling, memory bandwidth and Amdahl law. Confirm these hypotheses against measured timings.')

## Discussion for report (write after full measurements)

- Verify all four methods agree numerically on real data.
- Compare measured speedups at shared workload shapes, not extrapolated numbers.
- Explain why NumPy wins or loses compared with compiled Numba.
- Discuss parallel overhead and Amdahl's law, including that small matrices may not benefit.
- Note that Gram-matrix computation is a controlled proxy for vectorization, not an end-to-end semantic search benchmark.
- Record CPU details, BLAS/Numba thread settings, warm-up and skipped slow Python cases.
- Keep actual PNGs on Drive and mirror raw CSV and figure links to GitHub after execution.